In [1]:
import pandas as pd
import numpy as np
import os

# Настройки путей
INPUT_FILE = '../data/processed/train_cleaned.csv'
OUTPUT_FILE = '../data/processed/train_final.csv'

def create_custom_features(df):
    """
    Создание новых признаков (Feature Engineering).
    Эта же логика должна быть в веб-приложении!
    """
    # Копируем для безопасности
    df = df.copy()
    
    # --- Базовая очистка перед расчетами ---
    # DAYS_BIRTH и DAYS_EMPLOYED в исходных данных отрицательные (дни назад).
    # Делаем их положительными для удобства расчетов.
    if 'DAYS_BIRTH' in df.columns:
        df['DAYS_BIRTH'] = df['DAYS_BIRTH'].abs()
    
    if 'DAYS_EMPLOYED' in df.columns:
        df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].abs()
    
    # --- Генерация новых признаков ---
    
    # 1. Кредитная нагрузка: какой % дохода уходит на платеж
    # (Ожидаем: чем меньше, тем лучше)
    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    
    # 2. Платежный рейт: какой % от всей суммы кредита мы платим в год/месяц
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    
    # 3. Разница товаров: Насколько сумма кредита отличается от цены товара
    # (иногда люди берут кредит больше цены товара, чтобы получить наличные ("cash out"))
    if 'AMT_GOODS_PRICE' in df.columns:
        df['GOODS_DIFF'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    
    # 4. Процент жизни, отданный работе: (Стаж / Возраст)
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    
    # 5. Средний внешний рейтинг (Самый сильный признак!)
    # Считаем среднее по строке (axis=1), игнорируя NaN
    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    # Проверяем, есть ли эти колонки (на случай если какие-то удалили при чистке)
    existing_ext_cols = [c for c in ext_cols if c in df.columns]
    
    if existing_ext_cols:
        df['EXT_SOURCES_MEAN'] = df[existing_ext_cols].mean(axis=1)
    
    # --- Пост-обработка ---
    # Заменяем бесконечности на NaN (если вдруг поделили на 0)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    return df

if __name__ == "__main__":
    print(f"Загрузка данных из {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)
    
    print(f"Исходный размер: {df.shape}")
    
    print("Создание новых признаков...")
    df_final = create_custom_features(df)
    
    print(f"Сохранение в {OUTPUT_FILE}...")
    df_final.to_csv(OUTPUT_FILE, index=False)
    
    print(f"Готово! Итоговая форма: {df_final.shape}")
    
    # Проверка (выведем первые 5 значений нового признака)
    print("\nПример нового признака PAYMENT_RATE:")
    print(df_final['PAYMENT_RATE'].head())

Загрузка данных из ../data/processed/train_cleaned.csv...
Исходный размер: (307507, 106)
Создание новых признаков...
Сохранение в ../data/processed/train_final.csv...
Готово! Итоговая форма: (307507, 111)

Пример нового признака PAYMENT_RATE:
0    0.060749
1    0.027598
2    0.050000
3    0.094941
4    0.042623
Name: PAYMENT_RATE, dtype: float64
